In [1]:
import pandas as pd
raw = pd.read_csv("../data/raw/messy_ecommerce_sales_data.csv")
raw.shape

(103, 11)

# Cleaning messy e-commerce sales data
Goal: find and fix data quality problems, and document every decision.

In [2]:
import pandas as pd
import numpy as np

In [3]:
raw = pd.read_csv("../data/raw/messy_ecommerce_sales_data.csv")
df = raw.copy()

In [4]:
print(df.shape)
print(df.columns.tolist())
df.info()

(103, 11)
['ID', ' Customer_Name', 'Order_ID', 'Order_Date', 'Product', ' Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total']
<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              103 non-null    int64  
 1    Customer_Name  103 non-null    str    
 2   Order_ID        103 non-null    str    
 3   Order_Date      103 non-null    str    
 4   Product         103 non-null    str    
 5    Category       95 non-null     str    
 6   Quantity        98 non-null     str    
 7   Price           98 non-null     str    
 8   Payment_Method  103 non-null    str    
 9   Status          103 non-null    str    
 10  Total           89 non-null     float64
dtypes: float64(1), int64(1), str(9)
memory usage: 9.0 KB


In [5]:
df.head(10)

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,11/22/2024,Blender,Home,3,38,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,7/5/2025,Smartphone,Electronics,2,abd,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,12/23/2024,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,3/19/2025,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,10/20/2025,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51
5,105,Customer_105,ORD-22442,11/20/2024,Tennis Racket,Sports,3,122.06,Cash on Delivery,Cancelled,366.18
6,106,Customer_106,ORD-25885,2/2/2025,Blender,Home,NaN,978.63,Bank Transfer,Processing,NaN
7,107,Customer_107,ORD-89122,1/3/2025,Biography,Books,5,587.68,PayPal,Returned,2938.40
8,108,Customer_108,ORD-64400,10/23/2025,Science,Books,1,600.29,Cash on Delivery,Processing,600.29
9,109,Customer_109,ORD-18512,5/3/2025,Tennis Racket,Sports,5,168.34,Credit Card,Shipped,841.70


In [6]:
df.tail(5)

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
98,198,Customer_198,ORD-14608,7/27/2025,Vacuum,NaN,2,497.01,Cash on Delivery,Shipped,994.020
99,199,Customer_199,ORD-82922,1/22/2025,Blender,Home,5,372.28,Credit Card,Shipped,1861.400
100,175,Customer_175,ORD-56651,2/24/2025,Headphones,Electronics,1,111.36,Credit Card,Processing,77.952
101,142,Customer_142,ORD-69018,10/30/2025,Shoes,Clothing,5,645.26,Credit Card,Shipped,3226.300
102,146,Customer_146,ORD-32755,7/9/2025,Basketball,electronic,2,705.42,Bank Transfer,Processing,1410.840


In [7]:
df.columns = df.columns.str.strip().str.lower()
df.columns.tolist()

['id',
 'customer_name',
 'order_id',
 'order_date',
 'product',
 'category',
 'quantity',
 'price',
 'payment_method',
 'status',
 'total']

In [8]:
def quality_report(d):
    qty = pd.to_numeric(d["quantity"], errors="coerce")
    price = pd.to_numeric(d["price"], errors="coerce")
    dates = pd.to_datetime(d["order_date"], format="%m/%d/%Y", errors="coerce")
    q1, q3 = price.quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    expected_total = qty * price
    return pd.Series({
        "Rows": len(d),
        "Duplicate Order_IDs": int(d["order_id"].duplicated().sum()),
        "Missing cells": int(d.isna().sum().sum()),
        "Text inside Quantity/Price": int((qty.isna() & d["quantity"].notna()).sum() + (price.isna() & d["price"].notna()).sum()),
        "Zero/negative Quantity or Price": int(((qty <= 0) | (price <= 0)).sum()),
        "Price outliers (IQR rule)": int((price > upper).sum()),
        "Spellings of Category": int(d["category"].nunique()),
        "Spellings of Product": int(d["product"].nunique()),
        "Total does not match Qty x Price": int(((expected_total - d["total"]).abs() > 0.01).sum()),
        "Unreadable/missing dates": int(dates.isna().sum()),
    })

before = quality_report(df)
before

Rows                                103
Duplicate Order_IDs                   3
Missing cells                        32
Text inside Quantity/Price            5
Zero/negative Quantity or Price       3
Price outliers (IQR rule)             1
Spellings of Category                 9
Spellings of Product                 21
Total does not match Qty x Price      5
Unreadable/missing dates              2
dtype: int64

In [9]:
for col in ["customer_name", "order_id", "product", "category", "payment_method", "status"]:
    df[col] = df[col].str.strip()

df["product"] = df["product"].str.title()
df["category"] = df["category"].str.title().replace({"Electronic": "Electronics"})

df["category"].value_counts(dropna=False)

category
Books          22
Electronics    21
Home           20
Sports         17
Clothing       15
NaN             8
Name: count, dtype: int64

In [10]:
df[df["order_id"].duplicated(keep=False)].sort_values("order_id")

,id,customer_name,order_id,order_date,product,category,quantity,price,payment_method,status,total
46,146,Customer_146,ORD-32755,7/9/2025,Basketball,Electronics,2,705.42,Bank Transfer,Processing,1410.840
102,146,Customer_146,ORD-32755,7/9/2025,Basketball,Electronics,2,705.42,Bank Transfer,Processing,1410.840
75,175,Customer_175,ORD-56651,2/24/2025,Headphones,Electronics,1,111.36,Credit Card,Processing,111.360
100,175,Customer_175,ORD-56651,2/24/2025,Headphones,Electronics,1,111.36,Credit Card,Processing,77.952
42,142,Customer_142,ORD-69018,10/30/2025,Shoes,Clothing,5,645.26,Credit Card,Shipped,2258.410
101,142,Customer_142,ORD-69018,10/30/2025,Shoes,Clothing,5,645.26,Credit Card,Shipped,3226.300


In [11]:
print("exact duplicate rows:", df.duplicated().sum())
print("duplicate Order_IDs:", df["order_id"].duplicated().sum())

exact duplicate rows: 1
duplicate Order_IDs: 3


In [12]:
df = df.drop_duplicates(subset="order_id", keep="first").reset_index(drop=True)
len(df)

100

In [13]:
df["price"] = df["price"].str.replace("$", "", regex=False).replace({"four hundred": "400"})
df["quantity"] = df["quantity"].str.replace(r"[^0-9\-.]", "", regex=True)

df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df[["quantity", "price"]].dtypes

quantity    float64
price       float64
dtype: object

In [14]:
df.loc[df["quantity"] <= 0, "quantity"] = np.nan
df.loc[df["price"] <= 0, "price"] = np.nan

q1, q3 = df["price"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print("IQR upper limit:", upper)
df.loc[df["price"] > upper, ["id", "price"]]

IQR upper limit: 1427.9250000000002


,id,price
17,117,10000.0


In [15]:
df.loc[df["price"] > upper, "price"] = np.nan
df[["quantity", "price"]].isna().sum()

quantity    7
price       9
dtype: int64

In [16]:
d1 = pd.to_datetime(df["order_date"], format="%m/%d/%Y", errors="coerce")
d2 = pd.to_datetime(df["order_date"], format="%b %d %Y", errors="coerce")
df["order_date"] = d1.fillna(d2)

df["date_suspect"] = df["order_date"].isna() | (df["order_date"] < "2024-11-01")
df.loc[df["date_suspect"], ["id", "order_date"]]

,id,order_date
14,114,2023-01-05
45,145,2023-05-01
63,163,2023-01-05
92,192,NaT


In [17]:
lookup = (df.dropna(subset=["category"])
            .groupby("product")["category"]
            .agg(lambda s: s.mode().iloc[0]))
lookup

product
Basketball            Sports
Biography              Books
Blender                 Home
Comics                 Books
Fiction                Books
Football              Sports
Headphones       Electronics
Jacket              Clothing
Jeans               Clothing
Lamp                    Home
Laptop           Electronics
Microwave               Home
Science                Books
Shoes               Clothing
Smartphone       Electronics
Smartwatch       Electronics
T-Shirt             Clothing
Tennis Racket         Sports
Vacuum                  Home
Yoga Mat              Sports
Name: category, dtype: str

In [18]:
df["category"] = df["product"].map(lookup)
print(df["category"].isna().sum(), df["category"].nunique())

0 5


In [19]:
df["quantity_imputed"] = df["quantity"].isna()
df["price_imputed"] = df["price"].isna()

median_qty = df["quantity"].median()
median_price = df["price"].median()
print("median quantity:", median_qty, "| median price:", median_price)

df["quantity"] = df["quantity"].fillna(median_qty).round().astype(int)
df["price"] = df["price"].fillna(median_price).round(2)

print("quantity imputed:", df["quantity_imputed"].sum())
print("price imputed:", df["price_imputed"].sum())
print("rows with at least one imputed value:", (df["quantity_imputed"] | df["price_imputed"]).sum())

median quantity: 3.0 | median price: 531.06
quantity imputed: 7
price imputed: 9
rows with at least one imputed value: 14


In [20]:
df["total"] = (df["quantity"] * df["price"]).round(2)
df.loc[df["id"].isin([115, 142, 151, 185]), ["id", "quantity", "price", "total"]]

,id,quantity,price,total
15,115,2,796.84,1593.68
42,142,5,645.26,3226.30
51,151,2,457.16,914.32
85,185,2,730.92,1461.84


In [21]:
assert df["order_id"].is_unique
assert df["id"].is_unique
assert df[["quantity", "price", "total", "category", "product"]].notna().all().all()
assert (df[["quantity", "price", "total"]] > 0).all().all()
assert set(df["category"]) == {"Books", "Clothing", "Electronics", "Home", "Sports"}
assert ((df["quantity"] * df["price"]).round(2) == df["total"]).all()
print("All checks passed")

All checks passed


In [22]:
df[["quantity", "price", "total", "category", "product"]].isna().sum()

quantity    0
price       0
total       0
category    0
product     0
dtype: int64

In [23]:
after = quality_report(df)
pd.DataFrame({"Before": before, "After": after})

,Before,After
Rows,103,100
Duplicate Order_IDs,3,0
Missing cells,32,1
Text inside Quantity/Price,5,0
Zero/negative Quantity or Price,3,0
Price outliers (IQR rule),1,0
Spellings of Category,9,5
Spellings of Product,21,20
Total does not match Qty x Price,5,0
Unreadable/missing dates,2,1


In [24]:
df["order_month"] = df["order_date"].dt.strftime("%Y-%m")
df["counts_as_revenue"] = ~df["status"].isin(["Cancelled", "Returned"])

In [25]:
df.to_csv("../data/cleaned/clean_sales.csv", index=False)
df.shape

(100, 16)